<hr style="border: 6px solid#003262;" />

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="assets/content/images/01_cover.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

<br>

# IMAGE CLASSIFICATION AND RANDOM SEARCH

<br>

**About:** A hands-on guide to building and comparing three image classification approaches - Logistic Regression, a Fully Connected Neural Network, and a Convolutional Neural Network - then using Random Search to systematically find effective hyperparameters across all three.

**Learning Goals:**
* Load and prepare an image dataset for three different model types
* Understand why flattening, scaling, and PCA are necessary for classical models on image data
* Build a Logistic Regression classifier using PCA-reduced pixel features
* Build a Fully Connected Neural Network (FCNN) treating each pixel as an independent input
* Build a Convolutional Neural Network (CNN) that preserves and exploits spatial structure
* Apply Random Search to tune hyperparameters and understand why it outperforms grid search at scale
* Save model predictions for downstream use in ensemble selection

**Keywords:** image classification, convolutional neural network, fully connected network, logistic regression, PCA, random search, hyperparameter tuning, AUC

**Prerequisite Knowledge:** (1) Python and NumPy fundamentals, (2) Supervised learning basics - train/test splits, binary classification, (3) Familiarity with scikit-learn fit/predict interface

**Target User:** Intermediate ML practitioners who have fit sklearn models before and want to understand how neural network architectures differ from classical approaches, and how to automate hyperparameter selection across model types.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: DATASET AND SETUP](#Part_0)
> #### [PART 1: LOGISTIC REGRESSION WITH PCA](#Part_1)
> #### [PART 2: FULLY CONNECTED NEURAL NETWORK](#Part_2)
> #### [PART 3: CONVOLUTIONAL NEURAL NETWORK](#Part_3)
> #### [PART 4: RANDOM SEARCH FOR HYPERPARAMETER TUNING](#Part_4)

#### APPENDIX

> #### [APPENDIX A: SAVING MODEL PREDICTIONS](#Appendix_1)

<br>

In [ ]:
# All imports in the first cell - restart kernel and run from here to guarantee reproducibility.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform, randint

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)import os


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **DATASET** and Setup

<a id='Part_0_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 0.1: LOADING THE DIGITS DATASET

<br>

The `sklearn.digits` dataset contains 1,797 grayscale images of handwritten digits (0 through 9), each stored as an 8x8 grid of pixel intensities ranging from 0 to 16. This gives us 64 features per image - small enough to train quickly on a CPU, but large enough to show meaningful differences between model architectures.

We restrict the dataset to a binary classification problem (digit 0 vs. digit 1) throughout this notebook. Binary classification lets us use AUC as a single, interpretable metric and makes the ensemble notebook's workflow directly comparable.

___

**Note:** The underlying concept - comparing how LR, FCNN, and CNN handle the same image data - applies directly to larger image benchmarks such as MNIST (28x28 pixels, available via `keras.datasets.mnist`) or CIFAR-10 (32x32 color images). The architectural trade-offs explored here scale up without changing.

___

In [ ]:
# Load digits dataset and restrict to binary: digit 0 vs digit 1
digits = load_digits()

# Select only classes 0 and 1
mask = (digits.target == 0) | (digits.target == 1)
X_raw = digits.data[mask]         # shape: (n_samples, 64) - flattened 8x8 images
y = digits.target[mask]           # 0 or 1

# Visualize a few samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_raw[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample images from the digits dataset (0 vs 1)', y=1.02)
plt.tight_layout()
plt.show()

print(f'Dataset shape: {X_raw.shape}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

<a id='Part_0_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 0.2: TRAIN / VALIDATION / TEST SPLIT

<br>

A three-way split - train, validation, test - is essential for honest hyperparameter tuning. The validation set is used to compare model variants during training. The test set is held out and touched exactly once, at the very end, to estimate real-world performance.

Using the same test set to tune hyperparameters or pick models is **data leakage**: the model has seen the test set indirectly through the tuning loop, and reported accuracy will be optimistic.

___

**Note:** We use a stratified split (`stratify=y`) to ensure both classes appear in each split at their natural proportion. Without stratification, a small split could end up with only one class, making evaluation impossible.

___

In [ ]:
# First split: hold out 20% as the test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: from the remaining 80%, use 25% as validation (= 20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Validation: {X_val.shape[0]} samples')
print(f'Test: {X_test.shape[0]} samples')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **LOGISTIC REGRESSION** with PCA

<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: PREPARING IMAGE DATA FOR LOGISTIC REGRESSION

<br>

Logistic Regression expects a 1D feature vector per sample - it has no built-in notion of spatial structure. Each pixel is treated as an independent feature, regardless of which pixels are adjacent.

Two preprocessing steps are essential:

**Scaling:** Pixel values range 0 to 16. L-BFGS (our optimizer) uses gradient information; unscaled features cause gradients to be dominated by whichever feature has the largest magnitude. `StandardScaler` subtracts the mean and divides by standard deviation per feature, putting all pixels on comparable footing. Importantly, the scaler is fit on training data only - fitting on validation or test data would introduce leakage.

**PCA:** With 64 pixel features, LR can overfit on a small dataset. More importantly, adjacent pixels are highly correlated (similar intensities), so the feature matrix is low-rank. PCA finds orthogonal directions of maximum variance and projects the data onto them, removing redundant information. We retain enough components to explain 95% of the variance rather than fixing a component count - this adapts automatically to the dataset.

This preprocessing is the classical computer vision pipeline pre-deep learning. Notice what it replaces: PCA manually builds a feature extractor by looking at global pixel covariance. A CNN learns a task-specific spatial feature extractor from the training labels. Both reduce the raw pixel space to something more tractable, but through fundamentally different mechanisms.

___

**Sources Consulted:**
- [scikit-learn: LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) - API reference, parameter defaults verified against sklearn 1.3, 2026-08
- [scikit-learn: PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

___

In [ ]:
# Build a pipeline: Scale -> PCA -> Logistic Regression
# Using a Pipeline prevents data leakage: fit() on the pipeline applies
# all transformers and the final estimator only on the training fold.
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=42)),  # retain 95% variance
    ('clf', LogisticRegression(
        max_iter=1000,    # increase from default 100 - digits needs more iterations
        random_state=42,
        # TODO: verify solver default against current sklearn docs at https://scikit-learn.org
        solver='lbfgs'
    ))
])

# Fit on training data only
lr_pipeline.fit(X_train, y_train)

# How many PCA components did we keep?
n_components = lr_pipeline.named_steps['pca'].n_components_
print(f'PCA retained {n_components} components (95% variance explained)')

<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: EVALUATING THE LOGISTIC REGRESSION MODEL

<br>

We evaluate using **AUC (Area Under the ROC Curve)** rather than accuracy. AUC measures the model's ability to rank a random positive example (digit 1) above a random negative example (digit 0) across all possible decision thresholds. An AUC of 1.0 is perfect; 0.5 is a random classifier.

AUC is threshold-independent, which makes it the right metric when we want to compare models without committing to a specific operating point. In the ensemble notebook, all models' predicted probabilities are combined by averaging - so what matters is calibration quality (does a 0.8 probability actually indicate more confidence than a 0.6?) not just which side of 0.5 the prediction falls on.

In [ ]:
# Get predicted probabilities on validation set
# [:, 1] selects the probability for class 1 (digit 1)
val_probs_lr = lr_pipeline.predict_proba(X_val)[:, 1]
test_probs_lr = lr_pipeline.predict_proba(X_test)[:, 1]

val_auc_lr = roc_auc_score(y_val, val_probs_lr)
test_auc_lr = roc_auc_score(y_test, test_probs_lr)
print(f'Logistic Regression - Validation AUC: {val_auc_lr:.4f}')
print(f'Logistic Regression - Test AUC: {test_auc_lr:.4f}')

# Plot the ROC curve on validation data
fpr, tpr, _ = roc_curve(y_val, val_probs_lr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'LR + PCA (AUC={val_auc_lr:.3f})', color='#003262')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The Pipeline above fits the scaler, PCA, and classifier together. Why would fitting the scaler on the full dataset (X_raw) before creating the train/val split produce a misleading AUC estimate? Write code that demonstrates the data leakage by computing AUC both ways.**

<br>

```python
# Hint: fit a scaler on X_raw, transform X_train and X_val, then fit LR on the transformed train set.
# Compare the validation AUC to the pipeline AUC above.
from sklearn.preprocessing import StandardScaler
leaky_scaler = ...
# Your code here
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **FULLY CONNECTED** Neural Network

<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: ARCHITECTURE AND TRAINING

<br>

A Fully Connected Neural Network (FCNN) - also called a Dense or Multi-Layer Perceptron - connects every neuron in one layer to every neuron in the next. Like Logistic Regression, it treats each pixel as an independent input with no spatial structure built into the architecture. The key difference is depth: multiple non-linear layers allow the network to learn complex feature combinations that a single linear layer cannot represent.

**Rescaling:** Neural networks are sensitive to input scale. We rescale pixel values from [0, 16] to [0, 1] using a Keras `Rescaling` layer rather than a separate sklearn scaler. This keeps the entire preprocessing pipeline inside the Keras model, which simplifies deployment - a saved model can accept raw pixels without needing a separately saved scaler object.

**ReLU activation:** `relu(x) = max(0, x)` is the default activation in modern dense networks. It avoids the vanishing gradient problem that hampered sigmoid and tanh activations in deeper networks, and is computationally inexpensive.

**Dropout:** During training, dropout randomly zeroes a fraction of a layer's outputs. This prevents co-adaptation - neurons learn to not rely on any specific other neuron being active - which acts as implicit regularization. Dropout is only active during `model.fit()`; it is disabled during `model.predict()`.

**Binary cross-entropy loss:** Our output is a single sigmoid neuron giving P(class=1). The loss is $-[y \log \hat{p} + (1-y) \log(1-\hat{p})]$, which penalizes confident wrong predictions more heavily than uncertain ones.

___

**Sources Consulted:**
- [Keras: Dense layer](https://keras.io/api/layers/core_layers/dense/) - verified 2026-08
- Goodfellow et al., *Deep Learning* (2016) - Chapter 6: Deep Feedforward Networks (stable reference for ReLU and dropout rationale)

___

In [ ]:
def build_fcnn(n_units=64, dropout_rate=0.3):
    # Fully connected network for binary image classification
    model = keras.Sequential([
        layers.Input(shape=(64,)),
        layers.Rescaling(1.0 / 16),   # normalize 0-16 pixels to 0-1
        layers.Dense(n_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(n_units // 2, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid')  # single output: P(digit=1)
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['AUC']
    )
    return model

fcnn = build_fcnn(n_units=64, dropout_rate=0.3)
fcnn.summary()

In [ ]:
# Train the FCNN
history_fcnn = fcnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=0
)

# Evaluate
val_probs_fcnn = fcnn.predict(X_val, verbose=0).flatten()
test_probs_fcnn = fcnn.predict(X_test, verbose=0).flatten()

val_auc_fcnn = roc_auc_score(y_val, val_probs_fcnn)
test_auc_fcnn = roc_auc_score(y_test, test_probs_fcnn)
print(f'FCNN - Validation AUC: {val_auc_fcnn:.4f}')
print(f'FCNN - Test AUC: {test_auc_fcnn:.4f}')

# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(history_fcnn.history['loss'], label='Train loss', color='#003262')
plt.plot(history_fcnn.history['val_loss'], label='Val loss', color='red', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy loss')
plt.title('FCNN Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The FCNN flattens the 8x8 image into a 64-element vector before processing it. Modify `build_fcnn` to add a third hidden layer with 16 units between the second hidden layer and the output. Then re-train and report whether validation AUC improves or degrades. What does this tell you about model capacity vs. dataset size?**

<br>

```python
def build_fcnn_deeper(n_units=64, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(64,)),
        layers.Rescaling(1.0 / 16),
        # Add your layers here
        ...
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **CONVOLUTIONAL** Neural Network

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: WHY SPATIAL STRUCTURE MATTERS

<br>

The FCNN treats pixel (0,0) and pixel (0,1) as completely unrelated inputs. Their spatial relationship - that they are adjacent and likely correlated - is invisible to the model. Every connection between the input and the first hidden layer is learned independently, with no sharing.

A **Convolutional Neural Network** imposes a different inductive bias: instead of fully connected weights, each layer applies small learned **filters** (also called kernels) that slide across the image. A 3x3 filter computes a weighted sum of each local 3x3 neighborhood - the same filter weights are applied at every position. This gives two advantages:

1. **Parameter efficiency:** A 3x3 filter has 9 weights, regardless of image size. A fully connected first layer on an 8x8 image has 64 weights per neuron, and image size quadratically grows the parameter count.
2. **Translation equivariance:** A filter that detects a diagonal edge at position (2,3) will also detect it at position (5,6) without needing to learn a separate set of weights. This is why CNNs generalize well to variations in the position of the object in the image.

**MaxPooling** reduces the spatial dimensions by taking the maximum activation in each small region. After a `MaxPooling2D` with pool size 2, an 8x8 feature map becomes 4x4. This makes the representation spatially coarser (less sensitive to exact pixel location) while keeping the strongest activations.

___

**Sources Consulted:**
- LeCun et al., "Gradient-based learning applied to document recognition" (1998) - original CNN paper; stable reference for convolutional filters and pooling
- [Keras: Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/) - API reference verified 2026-08

___

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: BUILDING AND TRAINING THE CNN

<br>

In [ ]:
def build_cnn(n_filters=16, dropout_rate=0.3):
    # CNN for 8x8 grayscale binary image classification
    model = keras.Sequential([
        layers.Input(shape=(8, 8, 1)),  # height, width, channels
        layers.Rescaling(1.0 / 16),
        layers.Conv2D(n_filters, kernel_size=3, padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=2),  # 8x8 -> 4x4
        layers.Conv2D(n_filters * 2, kernel_size=3, padding='same', activation='relu'),
        layers.Flatten(),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['AUC']
    )
    return model

# Reshape from (n, 64) to (n, 8, 8, 1) for the CNN
X_train_2d = X_train.reshape(-1, 8, 8, 1).astype('float32')
X_val_2d = X_val.reshape(-1, 8, 8, 1).astype('float32')
X_test_2d = X_test.reshape(-1, 8, 8, 1).astype('float32')

cnn = build_cnn(n_filters=16, dropout_rate=0.3)
cnn.summary()

In [ ]:
history_cnn = cnn.fit(
    X_train_2d, y_train,
    validation_data=(X_val_2d, y_val),
    epochs=30,
    batch_size=32,
    verbose=0
)

val_probs_cnn = cnn.predict(X_val_2d, verbose=0).flatten()
test_probs_cnn = cnn.predict(X_test_2d, verbose=0).flatten()

val_auc_cnn = roc_auc_score(y_val, val_probs_cnn)
test_auc_cnn = roc_auc_score(y_test, test_probs_cnn)
print(f'CNN - Validation AUC: {val_auc_cnn:.4f}')
print(f'CNN - Test AUC: {test_auc_cnn:.4f}')

# Compare all three models
models = ['Logistic Regression + PCA', 'FCNN', 'CNN']
val_aucs = [val_auc_lr, val_auc_fcnn, val_auc_cnn]
colors = ['#003262', '#3B7EA1', '#C4820E']

plt.figure(figsize=(7, 4))
bars = plt.bar(models, val_aucs, color=colors)
plt.ylim(0.9, 1.01)
plt.ylabel('Validation AUC')
plt.title('Model Architecture Comparison')
for bar, auc in zip(bars, val_aucs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{auc:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The CNN above uses `padding='same'`, which zero-pads the input so the convolution output has the same height and width as the input. Change it to `padding='valid'` (no padding), then trace through the dimensions: what is the shape of the feature map after the first Conv2D layer on an 8x8 input with a 3x3 kernel? What about after MaxPooling? Verify your answer by calling `cnn.summary()` on the rebuilt model.**

<br>

```python
cnn_valid = build_cnn(...)  # pass the right argument for 'valid' padding
# Then check cnn_valid.summary() to verify your dimension calculation
# Expected shape after first Conv2D (valid, 3x3 kernel, 8x8 input): ???
# Expected shape after MaxPooling2D (pool=2): ???
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **RANDOM SEARCH** for Hyperparameter Tuning

<a id='Part_4_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.1: WHY RANDOM SEARCH BEATS GRID SEARCH

<br>

Hyperparameters are settings that are not learned from data during training - the learning rate, number of layers, dropout rate, regularization strength. Choosing them well makes the difference between a model that generalizes and one that over- or under-fits.

**Grid search** evaluates every combination of a pre-specified set of values. If you have 4 hyperparameters each with 5 values, that is $5^4 = 625$ evaluations. This is tractable for two or three hyperparameters but scales exponentially.

**Random search** samples each hyperparameter independently from a specified distribution. The key insight from Bergstra and Bengio (2012) is that in most ML models, a small number of hyperparameters account for most of the variance in performance. Grid search wastes evaluations on the unimportant dimensions - if hyperparameter B barely matters, you evaluate every value of B for each value of A. Random search covers the important dimensions more efficiently because each sample is independent.

For practical purposes, Random Search is the right default when you have more than 2 or 3 hyperparameters, each with a continuous or large discrete range. Bayesian optimization (the next notebook) can do better still by using past evaluations to decide where to sample next - at the cost of more implementation complexity and overhead.

___

**Sources Consulted:**
- Bergstra, J. and Bengio, Y., "Random Search for Hyper-Parameter Optimization," JMLR (2012) - primary source for the efficiency argument; stable reference
- [scikit-learn: RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) - verified 2026-08

___

<a id='Part_4_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.2: APPLYING RANDOM SEARCH TO LOGISTIC REGRESSION

<br>

`RandomizedSearchCV` from sklearn applies cross-validation within the training set. We use `cv=3` (3-fold) to keep runtime reasonable. For each of `n_iter=20` randomly sampled hyperparameter combinations, the pipeline is fit 3 times on different train folds and evaluated on the corresponding validation fold. The mean cross-validated AUC determines which configuration wins.

Notice that the parameter distributions use `loguniform` for the regularization strength `C`. This samples values uniformly on a log scale (e.g., 0.001, 0.01, 0.1, 1, 10, 100 are equally likely to appear). When a hyperparameter spans multiple orders of magnitude, log-uniform sampling covers the range far more efficiently than linear uniform sampling.

In [ ]:
# Random search over the Logistic Regression pipeline
param_dist_lr = {
    'pca__n_components': [0.85, 0.90, 0.95, 0.99],
    'clf__C': loguniform(1e-3, 1e2),      # log-uniform over 3 orders of magnitude
    'clf__solver': ['lbfgs', 'saga'],
    # TODO: verify available solvers against current sklearn docs at https://scikit-learn.org
}

search_lr = RandomizedSearchCV(
    lr_pipeline,
    param_distributions=param_dist_lr,
    n_iter=20,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1  # use all available CPU cores
)

search_lr.fit(X_train_val, y_train_val)  # fit on combined train+val for the search

print(f'Best LR params: {search_lr.best_params_}')
print(f'Best LR CV AUC: {search_lr.best_score_:.4f}')

# Evaluate on held-out test set
best_lr_probs = search_lr.predict_proba(X_test)[:, 1]
best_lr_auc = roc_auc_score(y_test, best_lr_probs)
print(f'Best LR Test AUC: {best_lr_auc:.4f}')

<a id='Part_4_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 4.3: GENERATING AND SAVING ALL MODEL PREDICTIONS

<br>

The ensemble notebook (02) takes a matrix of model predictions - one column per model - and finds the best combination. To produce 15 predictions (5 per model type), we train each architecture 5 times with different random seeds. This introduces controlled diversity: models with the same architecture but different random initialization tend to make correlated but not identical errors, which is exactly the kind of diversity that makes averaging beneficial.

In [ ]:
# Generate 5 validation predictions per model type using different random seeds
val_predictions = {}
test_predictions = {}

# LR variants - vary PCA threshold
for i, n_comp in enumerate([0.85, 0.90, 0.93, 0.96, 0.99]):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_comp, random_state=i)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=i, solver='lbfgs'))
    ])
    pipeline.fit(X_train, y_train)
    val_predictions[f'LR_{i}'] = pipeline.predict_proba(X_val)[:, 1]
    test_predictions[f'LR_{i}'] = pipeline.predict_proba(X_test)[:, 1]
    print(f'LR_{i} val AUC: {roc_auc_score(y_val, val_predictions[f"LR_{i}"]):.4f}')

# FCNN variants - vary architecture
for i, (units, dr) in enumerate([(32, 0.2), (64, 0.3), (128, 0.3), (64, 0.4), (32, 0.5)]):
    tf.random.set_seed(i)
    m = build_fcnn(n_units=units, dropout_rate=dr)
    m.fit(X_train, y_train, epochs=30, batch_size=32, verbose=0)
    val_predictions[f'FC_{i}'] = m.predict(X_val, verbose=0).flatten()
    test_predictions[f'FC_{i}'] = m.predict(X_test, verbose=0).flatten()
    print(f'FC_{i} val AUC: {roc_auc_score(y_val, val_predictions[f"FC_{i}"]):.4f}')

# CNN variants - vary filter count
for i, (filters, dr) in enumerate([(8, 0.2), (16, 0.3), (32, 0.3), (16, 0.4), (8, 0.5)]):
    tf.random.set_seed(i + 100)
    m = build_cnn(n_filters=filters, dropout_rate=dr)
    m.fit(X_train_2d, y_train, epochs=30, batch_size=32, verbose=0)
    val_predictions[f'CN_{i}'] = m.predict(X_val_2d, verbose=0).flatten()
    test_predictions[f'CN_{i}'] = m.predict(X_test_2d, verbose=0).flatten()
    print(f'CN_{i} val AUC: {roc_auc_score(y_val, val_predictions[f"CN_{i}"]):.4f}')

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!---------------------------------------->

> **The models above are trained with different random seeds to create diversity. Compute the Pearson correlation matrix among the 15 models' validation predictions using `pd.DataFrame(val_predictions).corr()`. Which pair of models is most correlated? Which is least correlated? What does high correlation between two models imply for the value of including both in an ensemble?**

<br>

```python
# Your code here
corr_matrix = ...
print('Most correlated pair:', ...)
print('Least correlated pair:', ...)
```

<hr style="border: 2px solid#003262;" />

In [ ]:
# Save predictions to CSV for use in notebook 02
import os
os.makedirs('data', exist_ok=True)

pd.DataFrame(val_predictions).to_csv('data/validation_predictions.csv', index=False)
pd.DataFrame(test_predictions).to_csv('data/test_predictions.csv', index=False)
pd.Series(y_val.astype(int), name='label').to_csv('data/validation_labels.csv', index=False)
pd.Series(y_test.astype(int), name='label').to_csv('data/test_labels.csv', index=False)

print('Saved: data/validation_predictions.csv')
print('Saved: data/test_predictions.csv')
print('Saved: data/validation_labels.csv')
print('Saved: data/test_labels.csv')
print(f'Shape: {pd.DataFrame(val_predictions).shape} (samples x models)')

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!---------------------------------------->

<a id='Appendix_1'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX A

## **SAVING** Model Predictions

The CSVs written at the end of Part 4 are the interface between this notebook and the ensemble notebook. Each column is one model's predicted probability for the positive class (digit 1) on the same set of samples. The rows must be in the same order across all files.

If you want to add your own models to the ensemble, add a column to `validation_predictions.csv` and `test_predictions.csv` with any name - the ensemble notebook treats all columns identically.

<hr style="border: 6px solid#003262;" />